# AI Resume Intelligence — NLP Extraction Pipeline

This notebook demonstrates the Natural Language Processing (NLP) pipeline used to extract structured information from raw resume text. It mirrors the functionality in `backend/app/utils/nlp.py`.

**Core Capabilities:**
1. **Regex Extraction**: Extracts emails and phone numbers.
2. **Named Entity Recognition (NER)**: Uses `spaCy` (`en_core_web_sm`) to extract candidate Names, Organizations, and Locations.
3. **Skill Matching**: Tokenizes the resume and matches tokens against a predefined list of skills (simulating a database lookup).

In [1]:
!pip install spacy

In [ ]:
import re
import spacy

# Ensure the spaCy model is downloaded before loading
try:
    nlp = spacy.load("en_core_web_sm")
    print("spaCy model 'en_core_web_sm' loaded successfully.")
except OSError:
    import subprocess
    print("Downloading 'en_core_web_sm' model...")
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")
    print("spaCy model 'en_core_web_sm' downloaded and loaded successfully.")

spaCy model 'en_core_web_sm' loaded successfully.


### Step 1: Regex Extraction (Email & Phone)

In [ ]:
def extract_email_phone(text: str):
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    phone_pattern = r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}'
    
    emails = re.findall(email_pattern, text)
    phones = re.findall(phone_pattern, text)
    
    return {
        "email": emails[0] if emails else None,
        "phone": phones[0] if phones else None
    }

sample_text = """
John Doe
Software Engineer
Email: john.doe@example.com | Phone: (555) 123-4567
Location: New York, NY
"""

contact_info = extract_email_phone(sample_text)
print("Extracted Contact Info:", contact_info)

Extracted Contact Info: {'email': 'john.doe@example.com', 'phone': '(555) 123-4567'}


### Step 2: Named Entity Recognition (NER) with spaCy

In [ ]:
def extract_entities(text: str):
    doc = nlp(text)
    
    entities = {
        "names": [],
        "organizations": [],
        "locations": []
    }
    
    for ent in doc.ents:
        if ent.label_ == "PERSON" and ent.text.strip() not in entities["names"]:
            entities["names"].append(ent.text.strip())
        elif ent.label_ == "ORG" and ent.text.strip() not in entities["organizations"]:
            entities["organizations"].append(ent.text.strip())
        elif ent.label_ == "GPE" and ent.text.strip() not in entities["locations"]:
            entities["locations"].append(ent.text.strip())
            
    return entities

sample_resume = """
Jane Smith
Senior Data Scientist at Google
San Francisco, California
Previously worked at Microsoft and Amazon.
"""

entities_info = extract_entities(sample_resume)
print("Extracted Entities:")
for category, values in entities_info.items():
    print(f"- {category.capitalize()}: {values}")

Extracted Entities:
- Names: ['Jane Smith']
- Organizations: ['Microsoft', 'Amazon']
- Locations: ['San Francisco', 'California']


### Step 3: Skill Matching
This simulates the database matching in `nlp.py` by tokenizing words and checking against a predefined dictionary.

In [ ]:
mock_skills_db = {"python", "java", "c++", "sql", "pandas", "machine learning", "aws", "docker", "react"}

def extract_skills_mock(text: str, mock_db: set):
    # 1. Extract alphanumeric words/tokens from the resume
    resume_words = set(re.findall(r'\b[a-zA-Z0-9+#]+\b', text.lower()))
    
    # Note: 'machine learning' is a multi-word phrase, so simple word tokenization might miss it unless handled as n-grams.
    # For simplicity, we match single tokens here:
    matched_skills = [skill for skill in resume_words if skill in mock_db]
    
    # Check for multi-word skills manually
    for skill in mock_db:
        if " " in skill and skill in text.lower():
            if skill not in matched_skills:
                matched_skills.append(skill)
                
    return matched_skills

sample_tech_resume = """
Experienced in Python and Java. Built data pipelines using Pandas and SQL.
Deployed applications on AWS using Docker. Familiar with Machine Learning models.
"""

skills_found = extract_skills_mock(sample_tech_resume, mock_skills_db)
print("Skills Found:", skills_found)

Skills Found: ['sql', 'pandas', 'python', 'docker', 'java', 'aws', 'machine learning']


### Step 4: Full Pipeline Integration

In [ ]:
def parse_resume_pipeline(text: str):
    contact = extract_email_phone(text)
    entities = extract_entities(text)
    skills = extract_skills_mock(text, mock_skills_db)
    
    return {
        "contact": contact,
        "entities": {
            "name_guesses": entities["names"][:3], 
            "companies": entities["organizations"][:10],
            "locations": entities["locations"][:5]
        },
        "matched_skills": skills[:20]
    }

full_resume_text = """
Alice Wonderland
Email: alice@example.com | Phone: 987-654-3210
Location: Seattle, WA

Senior Data Engineer at Amazon
Skills: Python, SQL, Docker, AWS, React, Pandas
"""

parsed_data = parse_resume_pipeline(full_resume_text)

import json
print("Final Parsed Data:")
print(json.dumps(parsed_data, indent=2))

Final Parsed Data:
{
  "contact": {
    "email": "alice@example.com",
    "phone": "987-654-3210"
  },
  "entities": {
    "name_guesses": [
      "Alice Wonderland\nEmail",
      "Docker",
      "Pandas"
    ],
    "companies": [
      "Phone",
      "Amazon",
      "SQL",
      "AWS"
    ],
    "locations": [
      "Seattle",
      "React"
    ]
  },
  "matched_skills": [
    "sql",
    "pandas",
    "python",
    "docker",
    "aws",
    "react"
  ]
}
